# Exp01 — Normal (Vanilla) Full Fine-Tuning

**Baseline / primary comparison** for all later experiments that add infrastructure.
Full fine-tune of RETFound-DINOv2-MEH on the 4 DR grades (R0/R1/R2/R3A) with the
plainest possible recipe — no imbalance handling, no per-layer LR tricks.

**Exactly two functional changes vs the P2B recipe:**

| | P2B (recommended) | Exp01 (vanilla baseline) |
|---|---|---|
| Loss | Focal γ=2 + inverse-freq class weights | plain CrossEntropy (no weights) |
| Per-layer LR | LLRD decay 0.75 (early layers barely move) | uniform — every layer at BASE_LR |

Everything else is held identical to P2B for a clean comparison: 4 classes,
gradient checkpointing, batch 16 × accum 2, AdamW (WD 0.05), cosine LR + 5-epoch
warmup, early stopping on val AUROC, 5-fold patient-stratified CV, 5-model test
ensemble + Youden thresholds.

> NOTE: `FOCAL_GAMMA`, `LLRD_DECAY`, and `CLASS_WEIGHTS` remain defined in the config
> cell (P2B inheritance) but are **unused** here — the loss is plain CE and the
> optimizer is built with `decay=1.0` (uniform LR).

In [ ]:
import os, sys, json, copy, math, time
from pathlib import Path

# Reduce CUDA memory fragmentation (helps on 12 GB GPUs with large models)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix

os.chdir(os.path.dirname(os.path.abspath('phase2b_full_finetune.ipynb')))
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

# ── Shared config (same as Phase 1/2A) ────────────────────────────────────────
N_FOLDS     = 5
MAX_EPOCHS  = 50
PATIENCE    = 10
INPUT_SIZE  = 224
NUM_CLASSES = 4
CLASS_WEIGHTS = [1.0, 1.796, 10.8469, 17.502]   # recomputed inverse-freq on NEW cohort
SEED        = 42
CLASSES     = ['R0', 'R1', 'R2', 'R3A']

# ── Full fine-tuning specific config ─────────────────────────────────────────
BASE_LR      = 5e-5    # much lower than LP: backbone is sensitive
MIN_LR       = 1e-7
WARMUP_EPOCHS = 5      # ramp up LR before cosine decay
WEIGHT_DECAY = 0.05    # AdamW regularisation
LLRD_DECAY   = 0.75    # per-layer LR multiplier toward input
GRAD_CLIP    = 1.0     # prevents exploding gradients in large ViTs+
BATCH_SIZE   = 16      # physical batch per GPU step (reduced for 12 GB GPU)
ACCUM_STEPS  = 2       # gradient accumulation: effective batch = 16 × 2 = 32
                       # gradients are divided by ACCUM_STEPS so training dynamics
                       # are identical to a true batch of 32 images

# ── Focal loss ────────────────────────────────────────────────────────────────
FOCAL_GAMMA = 2.0      # same as Phase 2A

HF_REPO   = 'YukunZhou/RETFound_dinov2_meh'
HF_FILE   = 'RETFound_dinov2_meh.pth'
CV_OUTPUT = Path('output_dir/exp01_normal_finetuning_cv')
CV_OUTPUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: Full fine-tuning on CPU is extremely slow (hours per fold).')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(DEVICE)
    print(f'GPU: {props.name}  VRAM: {props.total_memory/1e9:.1f} GB')
print(f'Config: BASE_LR={BASE_LR}, LLRD={LLRD_DECAY}, FOCAL_GAMMA={FOCAL_GAMMA}')
print(f'Batch:  physical={BATCH_SIZE}  accum_steps={ACCUM_STEPS}  effective={BATCH_SIZE*ACCUM_STEPS}')

Working directory: /home/eth/Desktop/isaack/RETFound-main
Device: cuda
GPU: NVIDIA RTX A4000  VRAM: 16.7 GB
Config: BASE_LR=5e-05, LLRD=0.75, FOCAL_GAMMA=2.0
Batch:  physical=16  accum_steps=2  effective=32


In [2]:
# Vanilla baseline: loss is plain nn.CrossEntropyLoss (defined in the CV loop).
# No focal loss and no class weights — that is the whole point of this baseline.
print('Loss: plain CrossEntropyLoss (no focal, no class weights).')

Loss: plain CrossEntropyLoss (no focal, no class weights).


In [3]:
# ── Load splits (identical to Phase 1/2A) ─────────────────────────────────────
GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}

df_all = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)

df_cv   = df_all[df_all['split'].isin(['train', 'val'])].copy()
df_test = df_all[df_all['split'] == 'test'].copy()

pat_grade = df_cv.groupby('code')['grade_int'].max().reset_index()
pat_grade.columns = ['code', 'strat_grade']
patient_ids   = pat_grade['code'].values
patient_strat = pat_grade['strat_grade'].values

# Same SEED → same folds as Phase 1 and 2A (direct comparison valid)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_assignments = {}
for fold_idx, (_, val_pat_idx) in enumerate(skf.split(patient_ids, patient_strat)):
    for pid in patient_ids[val_pat_idx]:
        fold_assignments[pid] = fold_idx
pat_grade['fold'] = pat_grade['code'].map(fold_assignments)

df_cv = df_cv.reset_index(drop=True)
df_cv['cv_idx'] = df_cv.index

print(f'CV pool : {len(df_cv)} images | {len(patient_ids)} patients')
print(f'Test set: {len(df_test)} images | {df_test["code"].nunique()} patients')
print('Same folds as Phase 1/2A (SEED=42) — direct comparison valid.')

CV pool : 7495 images | 1824 patients
Test set: 1349 images | 323 patients
Same folds as Phase 1/2A (SEED=42) — direct comparison valid.


In [4]:
import argparse
from util.datasets import build_transform

_aug_args = argparse.Namespace(
    input_size=INPUT_SIZE, color_jitter=None,
    aa='rand-m9-mstd0.5-inc1', reprob=0.25, remode='pixel', recount=1,
)
train_transform = build_transform('train', _aug_args)
eval_transform  = build_transform('val',   _aug_args)

class RetinopathyDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self):  return len(self.records)
    def __getitem__(self, idx):
        path, label = self.records[idx]
        return self.transform(Image.open(path).convert('RGB')), label

def make_records(df_subset):
    return [(row.image_path, row.grade_int) for row in df_subset.itertuples()]

print('Dataset helpers ready.')

Dataset helpers ready.


/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Backbone loading — full fine-tune version

Only one line changes versus the linear probe setup:

```python
# Linear probe:   param.requires_grad = ('head' in name)  → only head trainable
# Full fine-tune: param.requires_grad = True               → everything trainable
```

Everything else (checkpoint loading, key remapping, head initialisation) is identical.

In [5]:
import timm
from huggingface_hub import hf_hub_download
from timm.layers import trunc_normal_

def load_backbone_fft(device, num_classes=NUM_CLASSES, seed=None):
    """Load RETFound-DINOv2 with ALL parameters trainable for full fine-tuning.

    Gradient checkpointing is enabled to reduce activation memory:
    instead of storing all 24 blocks' intermediate activations simultaneously,
    only block inputs are stored and activations are recomputed during backward.
    This cuts activation memory by ~10x at the cost of ~33% more compute.
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=True, img_size=INPUT_SIZE,
        num_classes=num_classes, drop_path_rate=0.2,
    )
    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
    ckpt      = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    ckpt_model = ckpt['teacher']
    ckpt_model = {k.replace('backbone.', ''): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w12.', 'mlp.fc1.'): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w3.',  'mlp.fc2.'): v for k, v in ckpt_model.items()}
    state = model.state_dict()
    drop  = [k for k in ckpt_model if k in state and ckpt_model[k].shape != state[k].shape]
    for k in drop:
        print(f'  Skipping mismatched: {k}')
        del ckpt_model[k]
    model.load_state_dict(ckpt_model, strict=False)
    trunc_normal_(model.head.weight, std=2e-5)
    nn.init.zeros_(model.head.bias)
    # Full fine-tune: ALL parameters are trainable
    for param in model.parameters():
        param.requires_grad = True
    # Gradient checkpointing: recompute activations during backward to save memory
    model.set_grad_checkpointing(enable=True)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
    print('Gradient checkpointing: ENABLED')
    return model.to(device)

print('Verifying backbone load...')
_m = load_backbone_fft(DEVICE)
print('OK.')
del _m
torch.cuda.empty_cache()

Verifying backbone load...


  Skipping mismatched: pos_embed
Trainable: 303,232,004 / 303,232,004 (100.00%)
Gradient checkpointing: ENABLED


OK.


## Optimizer — uniform learning rate (LLRD disabled)

P2B uses layer-wise LR decay (LLRD): the head trains at `BASE_LR` and each layer
toward the input is scaled by `0.75`, so early blocks barely move. This vanilla
baseline **disables** that — we reuse the same `build_llrd_optimizer` helper but call
it with `decay=1.0`, so `BASE_LR × 1.0^depth = BASE_LR` for every parameter group.
All layers therefore train at the same learning rate.

The cosine schedule + 5-epoch warmup still scale every group together each epoch —
only the *per-layer* differentiation is removed. Bias/norm/positional params keep
weight-decay = 0 (standard ViT practice; not part of the baseline's two changes).

In [6]:
def build_llrd_optimizer(model, base_lr, weight_decay, decay=LLRD_DECAY):
    """
    AdamW with layer-wise learning rate decay.
    Each param is matched by name to a depth from the head;
    LR = base_lr × decay^depth.
    Bias, norm, cls_token, pos_embed: no weight decay.
    """
    num_blocks = len(model.blocks)

    def get_depth(name):
        if 'head' in name:
            return 0
        # Final norm (model.norm, not block norms)
        if name.startswith('norm'):
            return 1
        if 'blocks.' in name:
            block_idx = int(name.split('blocks.')[1].split('.')[0])
            # Last block (23) → depth 2; first block (0) → depth 25
            return num_blocks - block_idx + 1
        # patch_embed, cls_token, pos_embed, etc.
        return num_blocks + 2

    def no_decay(name):
        return any(tag in name for tag in ['bias', 'norm', 'cls_token', 'pos_embed'])

    # Group parameters by (depth, no_decay flag)
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        key = (get_depth(name), no_decay(name))
        groups.setdefault(key, []).append(param)

    param_groups = []
    for (depth, nd), params in sorted(groups.items()):
        lr = base_lr * (decay ** depth)
        param_groups.append({
            'params': params,
            'initial_lr': lr,   # stored so cosine schedule can scale it
            'lr': lr,
            'weight_decay': 0.0 if nd else weight_decay,
        })

    return torch.optim.AdamW(param_groups)


# Uniform-LR baseline: called below with decay=1.0, so every group = BASE_LR.
print(f'Uniform LR (no LLRD): all parameter groups at BASE_LR = {BASE_LR:.2e}')

Uniform LR (no LLRD): all parameter groups at BASE_LR = 5.00e-05


In [7]:
# ── Training helpers ──────────────────────────────────────────────────────────

class EarlyStoppingFFT:
    """
    Early stopping for full fine-tuning.
    Saves the full model state dict to disk (not just the head) because all
    307M parameters change during training.
    """
    def __init__(self, patience, model, checkpoint_path):
        self.patience        = patience
        self.best_auroc      = -float('inf')
        self.counter         = 0
        self.checkpoint_path = Path(checkpoint_path)
        torch.save(model.state_dict(), self.checkpoint_path)

    def step(self, auroc, model):
        if auroc != auroc: auroc = 0.0   # NaN guard
        if auroc > self.best_auroc:
            self.best_auroc = auroc
            self.counter    = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model, device):
        state = torch.load(self.checkpoint_path, map_location=device, weights_only=True)
        model.load_state_dict(state)


def get_lr(epoch, warmup_epochs, max_epochs, base_lr, min_lr):
    """Linear warmup then cosine decay."""
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * t))


def train_epoch_fft(model, loader, optimizer, criterion, device, scaler, epoch):
    """
    One training epoch for full fine-tuning with gradient accumulation.

    Gradient accumulation:
      ACCUM_STEPS mini-batches of size BATCH_SIZE are processed before taking
      one optimizer step. Loss is divided by ACCUM_STEPS so the effective gradient
      magnitude matches a single batch of size BATCH_SIZE × ACCUM_STEPS.

    Gradient checkpointing (set in load_backbone_fft):
      Activations are recomputed during backward instead of being stored —
      reduces peak activation memory ~10× at ~33% speed cost.

    Gradient clipping:
      Applied after unscaling so the clip threshold is in the original scale.
    """
    model.train()
    head_lr  = get_lr(epoch, WARMUP_EPOCHS, MAX_EPOCHS, BASE_LR, MIN_LR)
    lr_scale = head_lr / BASE_LR
    for pg in optimizer.param_groups:
        pg['lr'] = pg['initial_lr'] * lr_scale

    optimizer.zero_grad()
    total_loss   = 0.0
    n_samples    = 0
    step_count   = 0   # counts how many mini-batches since last optimizer step

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        # Determine if this is the last mini-batch overall
        is_last_batch = (i + 1 == len(loader))
        # Determine if we should take an optimizer step after this mini-batch
        should_step = ((step_count + 1) % ACCUM_STEPS == 0) or is_last_batch

        with torch.cuda.amp.autocast():
            # Divide loss by ACCUM_STEPS so accumulated gradient = true-batch gradient
            loss = criterion(model(imgs), labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS * len(labels)  # undo /ACCUM_STEPS for logging
        n_samples  += len(labels)
        step_count += 1

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step_count = 0

    return total_loss / n_samples, head_lr


@torch.no_grad()
def eval_fold(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().float()
        all_labels.append(labels)
        all_probs.append(probs)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()


def compute_metrics(labels, probs, preds=None):
    if preds is None:
        preds = probs.argmax(axis=1)
    probs_f64 = probs.astype(np.float64)
    probs_f64 = probs_f64 / probs_f64.sum(axis=1, keepdims=True)
    try:
        auroc = roc_auc_score(labels, probs_f64, multi_class='ovr', average='macro',
                              labels=list(range(NUM_CLASSES)))
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    cm    = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else float('nan'))
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))
    return {'auroc': auroc, 'accuracy': acc, 'kappa': kappa,
            'macro_sensitivity': np.nanmean(sens), 'macro_specificity': np.nanmean(spec),
            'sensitivity': np.array(sens), 'specificity': np.array(spec)}


print('Helpers defined.')

Helpers defined.


## 5-fold CV training loop — full fine-tuning

Differences from Phase 2A:
- `load_backbone_fft` instead of `load_backbone` (all params trainable)
- `build_llrd_optimizer` instead of plain `AdamW` (LLRD per layer)
- `train_epoch_fft` instead of `train_epoch` (LLRD schedule + grad clip)
- `EarlyStoppingFFT` instead of `EarlyStopping` (saves full model to disk)

In [8]:
criterion_cv  = nn.CrossEntropyLoss()  # vanilla baseline: no focal, no class weights

oof_labels_all = np.zeros(len(df_cv), dtype=np.int64)
oof_probs_all  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float32)
fold_results   = []

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{N_FOLDS}  [vanilla full fine-tune, plain CE]')
    print(f'{"="*60}')

    val_pats      = pat_grade[pat_grade['fold'] == fold]['code'].values
    train_pats    = pat_grade[pat_grade['fold'] != fold]['code'].values
    df_fold_train = df_cv[df_cv['code'].isin(train_pats)]
    df_fold_val   = df_cv[df_cv['code'].isin(val_pats)]
    val_cv_indices = df_fold_val['cv_idx'].values
    print(f'  Train: {len(df_fold_train)} images | Val: {len(df_fold_val)} images')

    ds_train = RetinopathyDataset(make_records(df_fold_train), train_transform)
    ds_val   = RetinopathyDataset(make_records(df_fold_val),   eval_transform)
    ds_test  = RetinopathyDataset(make_records(df_test),       eval_transform)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=12, pin_memory=True, drop_last=False)
    loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=12, pin_memory=True)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=12, pin_memory=True)

    model     = load_backbone_fft(device=DEVICE, seed=SEED + fold)
    optimizer = build_llrd_optimizer(model, BASE_LR, WEIGHT_DECAY, decay=1.0)  # uniform LR (no LLRD)
    scaler    = torch.cuda.amp.GradScaler()
    ckpt_path = CV_OUTPUT / f'best_fold_{fold}.pth'
    stopper   = EarlyStoppingFFT(patience=PATIENCE, model=model, checkpoint_path=ckpt_path)

    t_start = time.time()
    for epoch in range(MAX_EPOCHS):
        tr_loss, cur_lr = train_epoch_fft(
            model, loader_train, optimizer, criterion_cv, DEVICE, scaler, epoch
        )
        val_labels, val_probs = eval_fold(model, loader_val, DEVICE)
        m = compute_metrics(val_labels, val_probs)
        elapsed = time.time() - t_start
        print(f'  ep {epoch:02d} | lr(head)={cur_lr:.2e} | loss={tr_loss:.4f} | '
              f'AUROC={m["auroc"]:.4f} | sens={m["macro_sensitivity"]:.4f} | '
              f'elapsed={elapsed:.0f}s')
        if stopper.step(m['auroc'], model):
            print(f'  Early stop at epoch {epoch} (best AUROC={stopper.best_auroc:.4f})')
            break

    stopper.restore(model, DEVICE)
    print(f'  Best val AUROC: {stopper.best_auroc:.4f}')

    oof_labels, oof_probs = eval_fold(model, loader_val, DEVICE)
    oof_labels_all[val_cv_indices] = oof_labels
    oof_probs_all[val_cv_indices]  = oof_probs

    test_labels_fold, test_probs_fold = eval_fold(model, loader_test, DEVICE)

    np.save(CV_OUTPUT / f'fold_{fold}_oof_probs.npy',   oof_probs)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_labels.npy',  oof_labels)
    np.save(CV_OUTPUT / f'fold_{fold}_test_probs.npy',  test_probs_fold)
    np.save(CV_OUTPUT / f'fold_{fold}_test_labels.npy', test_labels_fold)

    m_fold = compute_metrics(oof_labels, oof_probs)
    fold_results.append({
        'fold': fold, 'best_auroc': stopper.best_auroc,
        'oof_auroc': m_fold['auroc'], 'oof_kappa': m_fold['kappa'],
        'oof_macro_sens': m_fold['macro_sensitivity'],
        'oof_macro_spec': m_fold['macro_specificity'],
    })
    print(f'  OOF AUROC: {m_fold["auroc"]:.4f}  Kappa: {m_fold["kappa"]:.4f}')

    del model
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('All folds complete.')
np.save(CV_OUTPUT / 'oof_labels_all.npy', oof_labels_all)
np.save(CV_OUTPUT / 'oof_probs_all.npy',  oof_probs_all)
with open(CV_OUTPUT / 'fold_results.json', 'w') as f:
    json.dump(fold_results, f, indent=2)
print(f'Saved to {CV_OUTPUT}/')


  FOLD 1/5  [vanilla full fine-tune, plain CE]
  Train: 6020 images | Val: 1475 images


  Skipping mismatched: pos_embed
Trainable: 303,232,004 / 303,232,004 (100.00%)
Gradient checkpointing: ENABLED


/tmp/ipykernel_222325/2108010784.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


/tmp/ipykernel_222325/2155862202.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

/tmp/ipykernel_222325/2155862202.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


  ep 18 | lr(head)=4.04e-05 | loss=0.5812 | AUROC=0.8962 | sens=0.6284 | elapsed=3427s
  Early stop at epoch 18 (best AUROC=0.9099)


  Best val AUROC: 0.9099


/tmp/ipykernel_222325/2155862202.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


  OOF AUROC: 0.9099  Kappa: 0.7404

All folds complete.
Saved to output_dir/exp01_normal_finetuning_cv/


## Test ensemble and Youden thresholds

In [9]:
from sklearn.metrics import roc_curve

# ── Test ensemble: average probabilities across the 5 fold models ─────────────
test_probs_list = [
    np.load(CV_OUTPUT / f'fold_{f}_test_probs.npy').astype(np.float64)
    for f in range(N_FOLDS)
]
test_labels_all = np.load(CV_OUTPUT / 'fold_0_test_labels.npy')
test_probs_ens  = np.mean(test_probs_list, axis=0)
test_probs_ens  = test_probs_ens / test_probs_ens.sum(axis=1, keepdims=True)

np.save(CV_OUTPUT / 'test_ensemble_probs.npy',  test_probs_ens)
np.save(CV_OUTPUT / 'test_ensemble_labels.npy', test_labels_all)
print('Test ensemble saved.')

# ── Youden thresholds on Phase 2B OOF ────────────────────────────────────────
p2b_oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy').astype(np.float64)
p2b_oof_probs  = p2b_oof_probs / p2b_oof_probs.sum(axis=1, keepdims=True)
p2b_oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')

youden_thr_p2b = []
for i in range(NUM_CLASSES):
    fpr, tpr, thrs = roc_curve((p2b_oof_labels == i).astype(int), p2b_oof_probs[:, i])
    j = tpr - fpr
    youden_thr_p2b.append(float(thrs[j.argmax()]))

print(f'Exp01 Youden thresholds: {[f"{t:.4f}" for t in youden_thr_p2b]}')

def apply_thresholds(probs, thresholds):
    thresholds = np.array(thresholds)
    ratios = probs / thresholds
    above  = probs > thresholds
    return np.where(above.any(axis=1), ratios.argmax(axis=1), probs.argmax(axis=1))

Test ensemble saved.
Exp01 Youden thresholds: ['0.6429', '0.3155', '0.0373', '0.0301']


## Results — full metric suite (accuracy, kappa, macro sens/spec, AUROC, confusion matrix)

In [10]:
# Self-contained results — full metric suite + confusion matrix. No cross-phase deps.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

EXP       = 'Exp01_Normal_finetuning'
EXP_LABEL = 'Exp01 Vanilla'
RECIPE    = 'vanilla 4-class full fine-tune (plain CE, uniform LR, no class weights)'
FIG_DIR   = Path('figures'); FIG_DIR.mkdir(exist_ok=True)


def _norm(p):
    p = p.astype(np.float64)
    return p / p.sum(axis=1, keepdims=True)


oof_lbl = np.load(CV_OUTPUT / 'oof_labels_all.npy')
oof_prb = _norm(np.load(CV_OUTPUT / 'oof_probs_all.npy'))
tst_lbl = np.load(CV_OUTPUT / 'test_ensemble_labels.npy')
tst_prb = _norm(np.load(CV_OUTPUT / 'test_ensemble_probs.npy'))


def print_table(title, items):
    print('\n' + '=' * 104)
    print(f'  {title}')
    print('=' * 104)
    hdr = (f'{"Configuration":<22} | {"Acc":>6} | {"AUROC":>6} | {"Kappa":>6} | '
           f'{"MacSens":>7} | {"MacSpec":>7} | ' + ' | '.join(f'{c:>6}' for c in CLASSES))
    print(hdr)
    print('-' * len(hdr))
    for name, lbl, prb, thr in items:
        preds = apply_thresholds(prb, thr) if thr is not None else None
        m = compute_metrics(lbl, prb, preds)
        s = m['sensitivity']
        print(f'{name:<22} | {m["accuracy"]:>6.4f} | {m["auroc"]:>6.4f} | {m["kappa"]:>6.4f} | '
              f'{m["macro_sensitivity"]:>7.4f} | {m["macro_specificity"]:>7.4f} | '
              + ' | '.join(f'{s[i]:>6.4f}' for i in range(NUM_CLASSES)))


print_table(f'{EXP_LABEL} — OOF', [
    (f'{EXP_LABEL} Argmax', oof_lbl, oof_prb, None),
    (f'{EXP_LABEL} Youden', oof_lbl, oof_prb, youden_thr_p2b),
])
print_table(f'{EXP_LABEL} — TEST (5-fold ensemble)', [
    (f'{EXP_LABEL} Argmax', tst_lbl, tst_prb, None),
    (f'{EXP_LABEL} Youden', tst_lbl, tst_prb, youden_thr_p2b),
])


def show_confusion(labels, probs, title, fname):
    preds = probs.argmax(1)
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    print(f'\nConfusion matrix — {title} (rows=true, cols=pred):')
    print('       ' + ' '.join(f'{c:>6}' for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f'{c:>6} ' + ' '.join(f'{cm[i, j]:>6d}' for j in range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES)
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    thr_c = cm.max() / 2 if cm.max() > 0 else 0.5
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thr_c else 'black')
    fig.colorbar(im, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, dpi=120); plt.close(fig)
    print(f'  saved {FIG_DIR / fname}')
    return cm


cm_oof  = show_confusion(oof_lbl, oof_prb, f'{EXP_LABEL} OOF',  f'{EXP}_cm_oof.png')
cm_test = show_confusion(tst_lbl, tst_prb, f'{EXP_LABEL} TEST', f'{EXP}_cm_test.png')


def _pack(lbl, prb, thr=None):
    preds = apply_thresholds(prb, thr) if thr is not None else None
    m = compute_metrics(lbl, prb, preds)
    return {'accuracy': m['accuracy'], 'auroc': m['auroc'], 'kappa': m['kappa'],
            'macro_sensitivity': m['macro_sensitivity'], 'macro_specificity': m['macro_specificity'],
            'sensitivity': m['sensitivity'].tolist(), 'specificity': m['specificity'].tolist()}


summary = {
    'experiment': EXP,
    'recipe': RECIPE,
    'classes': CLASSES,
    'base_lr': BASE_LR,
    'youden_thresholds': {c: youden_thr_p2b[i] for i, c in enumerate(CLASSES)},
    'oof':  {'Argmax': _pack(oof_lbl, oof_prb), 'Youden': _pack(oof_lbl, oof_prb, youden_thr_p2b)},
    'test': {'Argmax': _pack(tst_lbl, tst_prb), 'Youden': _pack(tst_lbl, tst_prb, youden_thr_p2b)},
    'confusion_matrix': {'oof': cm_oof.tolist(), 'test': cm_test.tolist()},
}
with open(CV_OUTPUT / 'exp01_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\nSummary saved to {CV_OUTPUT}/exp01_summary.json')



  Exp01 Vanilla — OOF
Configuration          |    Acc |  AUROC |  Kappa | MacSens | MacSpec |     R0 |     R1 |     R2 |    R3A
---------------------------------------------------------------------------------------------------------
Exp01 Vanilla Argmax   | 0.7805 | 0.8921 | 0.7222 |  0.5940 |  0.8931 | 0.9242 | 0.6157 | 0.4815 | 0.3546
Exp01 Vanilla Youden   | 0.7194 | 0.8921 | 0.7209 |  0.6340 |  0.8847 | 0.8591 | 0.4930 | 0.7457 | 0.4382

  Exp01 Vanilla — TEST (5-fold ensemble)
Configuration          |    Acc |  AUROC |  Kappa | MacSens | MacSpec |     R0 |     R1 |     R2 |    R3A
---------------------------------------------------------------------------------------------------------
Exp01 Vanilla Argmax   | 0.8258 | 0.9249 | 0.7798 |  0.6371 |  0.9125 | 0.9605 | 0.6627 | 0.5556 | 0.3696
Exp01 Vanilla Youden   | 0.7583 | 0.9249 | 0.7652 |  0.7084 |  0.9030 | 0.8926 | 0.4964 | 0.9444 | 0.5000

Confusion matrix — Exp01 Vanilla OOF (rows=true, cols=pred):
           R0     R1     